# Lecture 18: When the fastest motion is not unique

**The challenge.** Lecture 11 moved one axis, and the COPS robot arm, as fast
as possible. Lecture 14 moved a two-link arm with *minimum jerk*, not in
minimum time, and that choice was deliberate. Ask the natural question about
a machine with several axes, "what is the fastest way to get there?", and the
answer is a number but not a motion. One axis sets the time; the others have
time to spare, and any of their motions that finishes within that time is
just as fast. The necessary conditions cannot pick one, and shooting, which
needs an isolated solution, breaks down. This lecture shows exactly how, and
what it takes to ask a question that has one answer.

**After this lecture you can**

- recognize a problem whose optimal control is not unique, and say which
  part of the motion is undetermined;
- read the symptoms in the necessary conditions (vanishing costates, a
  switching function that is identically zero) and in shooting (a singular
  Jacobian);
- choose a well-posed formulation: a small secondary cost, a two-stage
  (lexicographic) solve, or a fixed path.

## 1. Synchronizing axes

Machine tools met this long before optimal control did. A numerically
controlled machine moving several axes at once does not run each axis as fast
as it can: the interpolator plans the slowest axis and *time-scales* the
others so that every axis starts and finishes together, which keeps the
cutter on the programmed line. The axes that could have gone faster simply
do not.

For robot arms the question was posed as an optimal control problem around
1970. Michael Kahn and Bernard Roth at Stanford studied the "near-minimum-time
control of open-loop articulated kinematic chains" (1971), and found that
the full problem, all joints free, was too hard to solve directly. The
decisive step came in 1985, from two groups independently: James Bobrow,
Steven Dubowsky and John Gibson at MIT, and Kang Shin and Neil McKay at
Michigan. Both *fixed the path* of the hand in advance and asked only how
fast to move along it. That reduces the problem to one scalar, the speed
along the path, and the time-optimal speed profile is found in its phase
plane, uniquely, riding the torque limits switch by switch. Fixing the path
is one way to remove the freedom this lecture is about; it is still how
most industrial controllers plan time-optimal motion.

## 2. Two axes, one deadline

The simplest case has no coupling at all: two jerk-limited axes,
$\dddot x_1 = j_1$ and $\dddot x_2 = j_2$ with $|j_i| \le J$, each moving
rest to rest, axis 1 over $D_1 = 10$ and axis 2 over $D_2 = 2$, both done by
the same final time, which is to be as small as possible. Alone, each axis
is Lecture 11's triple integrator: jerk $+J$, $-J$, $+J$ with switches at a
quarter and three quarters of the way, in time $T_i = 4\,(D_i / 2J)^{1/3}$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

from auto_optimal_control.homotopy import Homotopy
from auto_optimal_control.problem import Box, OptimalControlProblem

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

t = sp.Symbol("t", real=True)
x1, v1, a1, x2, v2, a2, j1, j2 = sp.symbols("x1 v1 a1 x2 v2 a2 j1 j2", real=True)
J, D1, D2, w = sp.symbols("J D1 D2 w", positive=True)


def two_axes(weight=False):
    """Two jerk-limited axes, minimum time; with weight, plus w int (j1^2 + j2^2) / 2."""
    cost = 1 + w * (j1**2 + j2**2) / 2 if weight else sp.S.One
    return OptimalControlProblem(
        name="two_axes" + ("_weighted" if weight else ""), time=t,
        states=[x1, v1, a1, x2, v2, a2], controls=[j1, j2],
        params=[J, D1, D2] + ([w] if weight else []), dynamics=[v1, a1, j1, v2, a2, j2],
        running_cost=cost, initial_state=[0] * 6,
        final_state={x1: D1, v1: 0, a1: 0, x2: D2, v2: 0, a2: 0},
        control_sets=[Box(j1, -J, J), Box(j2, -J, J)])


Jv, D1v, D2v = 1.0, 10.0, 2.0
T1, T2 = (4 * (D / (2 * Jv)) ** (1 / 3) for D in (D1v, D2v))
print(f"alone: axis 1 needs T1 = {T1:.6f}, axis 2 needs T2 = {T2:.6f}")

Both finish at $t_f = \max(T_1, T_2) = T_1$, with axis 1 doing exactly its
own bang-bang move. Axis 2 needs only $T_2 = 4$ of the $6.84$ available, and
*every* motion that gets it to $D_2$ at rest by $T_1$, within the jerk
limit, is optimal: the cost is the time, and the time is set by axis 1.
Here are three of them:

- its own bang-bang move in $T_2$, then waiting;
- the same bang-bang shape stretched to $T_1$, with the smaller jerk
  $J' = 32 D_2 / T_1^3$;
- the minimum-jerk quintic of Lecture 14 over $T_1$, whose peak jerk is
  $60 D_2 / T_1^3$.

In [ ]:
tt = np.linspace(0, T1, 2001)


def bang_bang(T, Jp):
    """Rest-to-rest triple integrator, jerk +Jp, -Jp, +Jp switching at T/4 and 3T/4, then
    at rest: position and jerk on tt, integrated piece by piece in closed form."""
    x, v, a, t0 = 0.0, 0.0, 0.0, 0.0
    pieces = []
    for t1, jp in ((T / 4, Jp), (3 * T / 4, -Jp), (T, Jp), (np.inf, 0.0)):
        pieces.append((t0, t1, x, v, a, jp))
        d = min(t1, 1e300) - t0
        if np.isfinite(t1):
            x, v, a = (x + v * d + a * d**2 / 2 + jp * d**3 / 6, v + a * d + jp * d**2 / 2,
                       a + jp * d)
        t0 = t1
    xs, js = np.zeros_like(tt), np.zeros_like(tt)
    for t0, t1, x0, v0, a0, jp in pieces:
        on = (tt >= t0) & (tt < t1)
        d = tt[on] - t0
        xs[on] = x0 + v0 * d + a0 * d**2 / 2 + jp * d**3 / 6
        js[on] = jp
    return xs, js


s = tt / T1
motions = {
    "its own S-curve, then wait": bang_bang(T2, 32 * D2v / T2 ** 3),
    "stretched bang-bang": bang_bang(T1, 32 * D2v / T1 ** 3),
    "minimum-jerk quintic": (D2v * (10 * s**3 - 15 * s**4 + 6 * s**5),
                             D2v / T1**3 * (60 - 360 * s + 360 * s**2)),
}
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.4))
for (name, (x, j)), color in zip(motions.items(), [BLUE, ORANGE, GRAY]):
    left.plot(tt, x, color=color, lw=2, label=name)
    right.plot(tt, j, color=color, lw=1.6)
    print(f"{name:28s} ends at x = {x[-1]:.4f}, peak |jerk| = {np.abs(j).max():.3f} <= J = {Jv}")
left.axhline(D2v, color="k", lw=0.8, ls="--")
left.set(xlabel="t", ylabel="axis 2 position", title="three optimal motions of axis 2")
right.set(xlabel="t", ylabel="axis 2 jerk", title="their controls")
left.legend(frameon=False, fontsize=8)
for ax in (left, right):
    ax.grid(alpha=0.3)
fig.tight_layout()

In [ ]:
# Three XY gantries, one per optimal motion of axis 2 (the carriage on the bridge), with
# axis 1 (the bridge) on its bang-bang move: drawn every T1/8. All arrive together.
from matplotlib.patches import Rectangle

x_axis1, _ = bang_bang(T1, Jv)                 # axis 1: its own time-optimal move over T1
frames = np.linspace(0, len(tt) - 1, 9).astype(int)
fig, axes = plt.subplots(3, 1, figsize=(10, 7.2), sharex=True)
for ax, (name, (y_axis2, _)), color in zip(axes, motions.items(), [BLUE, ORANGE, GRAY]):
    ax.add_patch(Rectangle((-0.6, -0.5), 11.2, 0.25, color="k", alpha=0.15, lw=0))
    ax.add_patch(Rectangle((-0.6, 2.25), 11.2, 0.25, color="k", alpha=0.15, lw=0))
    ax.plot(x_axis1, y_axis2, color=color, lw=1.5, alpha=0.6)
    for n, i in enumerate(frames):
        fade = 0.2 + 0.8 * n / (len(frames) - 1)
        xb, yc = x_axis1[i], y_axis2[i]
        ax.plot([xb, xb], [-0.4, 2.4], color="k", lw=3, alpha=fade * 0.5, solid_capstyle="butt")
        ax.add_patch(Rectangle((xb - 0.25, yc - 0.12), 0.5, 0.24, color=color, alpha=fade, lw=0))
    ax.plot(10, 2, "*", color="k", ms=14, zorder=5)
    ax.text(10.1, 2.05, "target", fontsize=8)
    ax.set_title(name, fontsize=10, loc="left")
    ax.set(xlim=(-0.8, 11.3), ylim=(-0.7, 2.7), ylabel="axis 2")
    ax.set_aspect("equal")
axes[-1].set_xlabel("axis 1")
fig.suptitle("three XY gantries: the bar is the bridge (axis 1, along the rails), the block the "
             "carriage on it (axis 2)\n"
             f"drawn every T1/8, darker = later; all three reach the target at t = T1 = {T1:.3f}",
             fontsize=10, x=0.01, ha="left")
fig.tight_layout()

*The same three motions on a machine: an XY gantry whose bridge (axis 1) runs along the rails on its time-optimal move, carrying the carriage (axis 2). Each gantry takes the same time; only the carriage's path differs.*

In [ ]:
# The same three gantries, moving: press play.
from IPython.display import HTML
from matplotlib import animation
from matplotlib.patches import Rectangle

plt.rcParams["animation.embed_limit"] = 30
x_axis1, _ = bang_bang(T1, Jv)
n_frames = 60
idx = np.linspace(0, len(tt) - 1, n_frames).astype(int)
fig, axes = plt.subplots(3, 1, figsize=(7, 5.4), sharex=True, dpi=72)
bridges, carriages, trails, clock = [], [], [], None
for ax, (name, (y_axis2, _)), color in zip(axes, motions.items(), [BLUE, ORANGE, GRAY]):
    ax.add_patch(Rectangle((-0.6, -0.5), 11.2, 0.25, color="k", alpha=0.15, lw=0))
    ax.add_patch(Rectangle((-0.6, 2.25), 11.2, 0.25, color="k", alpha=0.15, lw=0))
    ax.plot(10, 2, "*", color="k", ms=10)
    bridges.append(ax.plot([], [], color="k", lw=3, alpha=0.5, solid_capstyle="butt")[0])
    carriages.append(ax.add_patch(Rectangle((0, 0), 0.5, 0.24, color=color, lw=0)))
    trails.append(ax.plot([], [], color=color, lw=1.2, alpha=0.6)[0])
    ax.set_title(name, fontsize=9, loc="left")
    ax.set(xlim=(-0.8, 11.3), ylim=(-0.7, 2.7), xticks=[], yticks=[])
    ax.set_aspect("equal")
clock = axes[0].text(11.2, 2.75, "", fontsize=9, va="bottom", ha="right")


def draw(k):
    i = idx[k]
    for (_, (y_axis2, _)), b, c, tr_ in zip(motions.items(), bridges, carriages, trails):
        b.set_data([x_axis1[i]] * 2, [-0.4, 2.4])
        c.set_xy((x_axis1[i] - 0.25, y_axis2[i] - 0.12))
        tr_.set_data(x_axis1[:i + 1], y_axis2[:i + 1])
    clock.set_text(f"t = {tt[i]:.2f} of {T1:.2f}")
    return bridges + carriages + trails + [clock]


anim = animation.FuncAnimation(fig, draw, frames=n_frames, interval=80, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())

*The three gantries moving together. Press play: the bridges move identically, the carriages differently, and all three arrive at the same moment.*

## 3. What the necessary conditions say

The Hamiltonian separates into one part per axis:

$$
H = 1 + \big(\lambda_{x_1} v_1 + \lambda_{v_1} a_1 + \lambda_{a_1} j_1\big)
      + \big(\lambda_{x_2} v_2 + \lambda_{v_2} a_2 + \lambda_{a_2} j_2\big),
$$

and so do the costate equations. With axis 1's costates from its own
time-optimal solution and *all of axis 2's costates zero*, every condition
holds: the costate equations (zero stays zero), $H = 0$ along the way, and
the transversality conditions (the terminal multipliers of axis 2 are zero
too). Axis 2's switching function is then $S_2 = \lambda_{a_2} \equiv 0$,
and the minimum principle, "minimize $S_2\, j_2$ over $|j_2| \le J$", says
nothing at all: every $j_2$ minimizes. The conditions are satisfied by all
of axis 2's motions at once.

That is what non-uniqueness looks like from the inside: a costate block that
vanishes and a switching function that is identically zero on the whole
horizon. (It resembles a singular arc, Lecture 12, but differentiating
$S_2 \equiv 0$ gives nothing either: $\lambda_{a_2}$ and all its derivatives
are zero, so no singular control can be solved for.) Shooting needs the
solution to be an *isolated* root of the residual, and here there is a
whole family; the Jacobian is singular there. The automatic solve shows the
symptom: it finds the time, from the smoothed problem, but no exact
extremal.

In [ ]:
h_pure = Homotopy(two_axes())
r_pure = h_pure.solve([Jv, D1v, D2v], ([-3] * 6, [3] * 6), (2.0, 20.0))
print(r_pure.exact, "-", r_pure.message)
print(f"t_f of the last smoothed solution: {r_pure.solution.tf:.6f}  (T1 = {T1:.6f})")

## 4. Asking for one answer

The cure is to say which of the equally fast motions we want. The most
common choice adds a small secondary cost, here jerk energy:

$$
\min\; t_f + w \int_0^{t_f} \tfrac12\big(j_1^2 + j_2^2\big)\,dt .
$$

Now $H$ is strictly convex in both jerks, the control law is a clipped
linear one (Lecture 4) and the solution is unique. As $w \to 0$, the time
approaches $T_1$, axis 1 approaches its bang-bang move, and axis 2 is left
to minimize its own jerk energy in the time it is given: the quintic. Watch
the slack axis's costates and the conditioning of the shooting Jacobian on
the way:

In [ ]:
from auto_optimal_control.shooting import SingleShooting

h_w = Homotopy(two_axes(weight=True))
quintic = lambda tr: D2v * (10 * (tr["t"] / tr["t"][-1]) ** 3 - 15 * (tr["t"] / tr["t"][-1]) ** 4
                            + 6 * (tr["t"] / tr["t"][-1]) ** 5)
results = {}
print("     w        t_f      max|lam axis 2|  cond(dR/dv)   axis 2 - quintic")
for wv in (1.0, 0.1, 0.01, 0.001):
    rw = h_w.solve([Jv, D1v, D2v, wv], ([-3] * 6, [3] * 6), (2.0, 20.0))
    results[wv] = rw
    _, Jac = rw.shooting.residual_jacobian(rw.solution.v)
    sv = np.linalg.svd(Jac, compute_uv=False)
    tr = rw.shooting.trajectory(rw.solution.v, n_points=2001)
    print(f"{wv:7.0e}  {rw.solution.tf:.6f}   {np.abs(rw.solution.lam0[3:]).max():.2e}"
          f"        {sv[0] / sv[-1]:.1e}      {np.abs(tr['z'][:, 3] - quintic(tr)).max():.1e}")

Three things to read off the table:

- **The time converges to $T_1$**, from 7.68 at $w = 1$ to within $10^{-6}$
  at $w = 10^{-3}$.
- **Axis 2's costates shrink in proportion to $w$.** In the limit they
  vanish, as section 3 found; the penalty is what keeps them, and with
  them axis 2's motion, determined.
- **The condition number grows like $1/w$.** The family of section 3 is
  coming back: as the penalty fades, the residual becomes flat in the
  directions that move axis 2, and shooting has less and less to grip.

And at every $w$ axis 2 *is* the quintic, to rounding. The penalty does not
just break the tie; it breaks it the way we asked, by picking the smoothest
of the fastest motions.

In [ ]:
rw = results[1e-3]
tr = rw.shooting.trajectory(rw.solution.v, n_points=2001)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharex=True)
for i, (ax, name) in enumerate(zip(axes, ["axis 1 (sets the time)", "axis 2 (has time to spare)"])):
    ax.plot(tr["t"], tr["u"][:, i], color=BLUE if i == 0 else ORANGE, lw=2)
    ax.axhline(Jv, color="k", lw=0.8, ls="--")
    ax.axhline(-Jv, color="k", lw=0.8, ls="--")
    ax.set(xlabel="t", ylabel="jerk", title=name)
    ax.grid(alpha=0.3)
fig.tight_layout()

## 5. The price, and the choices

The penalty cannot be taken to zero. From random draws the solve already
fails at $w = 10^{-4}$; continuing in $w$ from the solution at $10^{-3}$
gets further, and then stalls as the residual flattens:

In [ ]:
from auto_optimal_control.shooting import parameter_continuation

r4 = h_w.solve([Jv, D1v, D2v, 1e-4], ([-3] * 6, [3] * 6), (2.0, 20.0))
print(f"w = 1e-4 from random draws: exact {r4.exact} - {r4.message}")
r3 = results[1e-3]
try:
    parameter_continuation(h_w.kr, r3.shooting.params, "w", np.logspace(-3, -6, 13),
                           r3.solution.v)
except RuntimeError as err:
    print("continuing in w from 1e-3:", err)
print(f"at w = 1e-3, t_f - T1 = {r3.solution.tf - T1:.1e}")

The exact limit, the bang-bang problem of section 3, is out of reach by
construction. So the choice of formulation is a real one, and there are
three standard answers:

1. **A small secondary cost**, as above: $t_f + w\int L$. Simple and
   well-posed, at the price of a time slightly above the minimum (the
   difference printed above) and a conditioning that degrades as $w$
   falls.
2. **A two-stage (lexicographic) solve.** First find the minimum time $T^*$
   (it *is* unique, even when the motion is not), then fix
   $t_f = (1 + \delta) T^*$ and minimize the secondary cost: the
   minimum-jerk problem of Lecture 14, with the jerk limits as control
   bounds. Each stage is well-posed; the second has room to choose only
   because $\delta > 0$.
3. **A fixed path.** Bobrow, Dubowsky and Gibson's and Shin and McKay's
   reduction: prescribe the geometric path and optimize only the speed
   along it, a one-dimensional problem whose time-optimal profile is
   unique.

All three appear in practice. For a robot arm with jerk, torque and speed
limits, the first is the natural setting for the package's next step:
minimize time plus a small jerk penalty, with bounded jerks and torque
limits, so that the joint that sets the pace is nearly bang-bang and the
others move smoothly, and every solve has one answer to converge to.

## Summary

- Minimum time on several axes fixes the *time* but not the *motion*: one
  axis sets the time, and every motion of the others that finishes within it
  is optimal.
- The symptoms: the slack axes' costates vanish, their switching functions
  are identically zero, the minimum principle says nothing about their
  controls, and the shooting Jacobian is singular.
- A small secondary cost $w\int L$ makes the solution unique and selects the
  smoothest of the fastest motions; the costates of the slack axes scale
  like $w$ and the conditioning like $1/w$, so $w$ cannot go to zero.
- The alternatives are a two-stage solve (minimum time, then minimum
  energy at a slightly longer time) and a fixed path (one-dimensional speed
  optimization).

## Exercises

1. Make axis 2 the one that sets the time (swap $D_1$ and $D_2$) and check
   that the roles swap in the table.
2. Choose $D_2$ so that $T_2 = T_1$. Is the solution of section 2 unique
   now? What happens to the costates of both axes?
3. Carry out the two-stage solve of section 5 with $\delta = 10^{-2}$, and
   compare its axis 1 with the bang-bang move and its axis 2 with the
   quintic.
4. Couple the axes: add a combined jerk limit $|j_1| + |j_2| \le J_{\text{sum}}$
   as a smooth path constraint on a new state, or a shared torque limit as in
   Lecture 14's arm. Does the coupling remove the freedom?

## Further reading

- M. E. Kahn and B. Roth, "The near-minimum-time control of open-loop
  articulated kinematic chains", *Journal of Dynamic Systems, Measurement,
  and Control* 93(3), 1971.
- J. E. Bobrow, S. Dubowsky and J. S. Gibson, "Time-optimal control of
  robotic manipulators along specified paths", *The International Journal of
  Robotics Research* 4(3), 1985.
- K. G. Shin and N. D. McKay, "Minimum-time control of robotic manipulators
  with geometric path constraints", *IEEE Transactions on Automatic
  Control* 30(6), 1985.
- T. Flash and N. Hogan, "The coordination of arm movements: an
  experimentally confirmed mathematical model", *Journal of Neuroscience*
  5(7), 1985: the minimum-jerk model of Lecture 14.